# Phase 6 — Weighted Temporal SHAP (project's own contribution)

Runs `src/explainability/weighted_temporal_shap.py`, then visualizes the ablation, stability comparison, and fairness-reduction results. See that script's module docstring for the full design rationale (cost-aware weighting per `docs/reading_material.md` §8.3 Option 1, chosen over adaptive window length; the SHAP-guided selective threshold correction mechanism; the small-eval-sample scale caveat).

**Running on Google Colab — read this before running anything:**

Setup is split into two cells (Part A, Part B) because installing this project's pinned `numpy` version mid-session conflicts with the numpy Colab's Python process already has loaded — Colab does **not** auto-restart after `pip install` the way you might expect, so a manual restart in between is required, or every later import silently breaks in confusing ways (`cannot import name '_center' from 'numpy._core.umath'` if you've seen that error, that's this).

1. Upload your 7 `HC_*.csv` files (already at `data/raw/` on this machine) to a Google Drive folder, e.g. `MyDrive/home-credit-data/`.
2. Run the **Part A** cell below (clones the repo, installs pinned packages).
3. **Runtime → Restart session** (top menu). This is required, not optional — do not skip it, and do not re-run Part A after restarting.
4. Run the **Part B** cell (mounts Drive, copies the data in, rebuilds the pipeline — ~25 min, mostly the SQL step).
5. Run the rest of the notebook from the `%run` cell onward.

Both Part A and Part B are safe to re-run if something fails partway — they check what already exists first rather than blindly re-cloning/re-copying/re-cding, which is what caused the earlier `.../credit-risk-scoring-26/credit-risk-scoring-26/...` nesting bug.

In [ ]:
# --- Colab setup, PART A: clone + install. Skip entirely if running locally with everything already built. ---
# Idempotent: safe to re-run (checks whether the repo already exists instead of re-cloning into itself).

import os

REPO_DIR = "/content/credit-risk-scoring-26"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AdityaMallaThakuri/credit-risk-scoring-26.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists -- skipping clone")

os.chdir(REPO_DIR)
!pip install -q -r requirements.txt

print(
    "\nPart A done.\n"
    ">>> Now: Runtime -> Restart session, then run the PART B cell below. <<<\n"
    ">>> Do NOT re-run this Part A cell after restarting -- go straight to Part B. <<<"
)

In [ ]:
# --- Colab setup, PART B: run this AFTER restarting the runtime post-Part-A. ---
# Idempotent: skips any raw file already copied, and skips the SQL pipeline
# rebuild if final_feature_table already exists in credit_risk.db.

import os
import shutil
import sqlite3
from pathlib import Path

REPO_DIR = "/content/credit-risk-scoring-26"
os.chdir(REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/home-credit-data'  # change this if you used a different folder name

os.makedirs("data/raw", exist_ok=True)

# Accepts either the original Kaggle filenames or this project's HC_*.csv names,
# whichever you uploaded to Drive -- copies + renames into data/raw/HC_*.csv.
rename_map = {
    "application_train.csv": "HC_application_train.csv",
    "bureau.csv": "HC_bureau.csv",
    "bureau_balance.csv": "HC_bureau_balance.csv",
    "credit_card_balance.csv": "HC_credit_card_balance.csv",
    "installments_payments.csv": "HC_installments_payments.csv",
    "POS_CASH_balance.csv": "HC_POS_CASH_balance.csv",
    "previous_application.csv": "HC_previous_application.csv",
}

drive_dir = Path(DRIVE_DATA_DIR)
missing = []
for original_name, hc_name in rename_map.items():
    dst = Path("data/raw") / hc_name
    if dst.exists():
        print(f"{dst} already present -- skipping")
        continue
    src_candidates = [drive_dir / hc_name, drive_dir / original_name]
    src = next((p for p in src_candidates if p.exists()), None)
    if src is None:
        missing.append(hc_name)
        continue
    shutil.copy(src, dst)
    print(f"Copied {src.name} -> {dst}")

if missing:
    raise FileNotFoundError(
        f"Missing from {DRIVE_DATA_DIR}: {missing}. "
        "Check the folder name/path and that all 7 files were uploaded."
    )

db_path = Path("data/processed/credit_risk.db")
need_sql_pipeline = True
if db_path.exists():
    conn = sqlite3.connect(db_path)
    tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    conn.close()
    need_sql_pipeline = "final_feature_table" not in tables

if need_sql_pipeline:
    !python src/features/run_sql_pipeline.py
else:
    print("final_feature_table already exists -- skipping run_sql_pipeline.py")

!python src/features/engineer_features.py
!python src/features/build_modeling_feature_set.py
!python src/features/build_synthetic_overlay.py
!python src/models/cv_split.py
!python src/explainability/adaptive_shap.py

os.chdir(f"{REPO_DIR}/notebooks")
print("\nPart B done -- now run the %run cell below.")

In [ ]:
%run ../src/explainability/weighted_temporal_shap.py

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

print("stability_df:", stability_df.shape)
print("fairness_df:", fairness_df.shape)
print("weight_log:", weight_log.shape)

## 1. w_drift vs w_cost — do statistically-drifted features and financially-consequential features agree?

This is the entire premise of cost-aware weighting: if these two rankings agreed, blending them would be pointless. Look for features that score very differently on the two axes (top-right or bottom-left = agreement; top-left/bottom-right = exactly where cost-aware weighting changes the explanation).

In [ ]:
latest_period = weight_log["period"].iloc[-1]
wl = weight_log[weight_log["period"] == "P4"].sort_values("w_cost", ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(wl["w_drift"], wl["w_cost"], alpha=0.7)
for _, row in wl.head(8).iterrows():
    ax.annotate(row["feature"], (row["w_drift"], row["w_cost"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("w_drift = 1 / (1 + PSI)  (higher = LESS statistically drifted)")
ax.set_ylabel("w_cost  (higher = MORE Expected-Loss-consequential)")
ax.set_title("P4: statistical drift-trust vs. financial-cost weight, per feature\n(top 8 by w_cost labeled)")
plt.tight_layout()
plt.show()

## 2. w_cost by period — does the counterfactual EL-sensitivity ranking shift as drift escalates?

In [ ]:
top_features_p4 = weight_log[weight_log["period"] == "P4"].nlargest(8, "w_cost")["feature"].tolist()
pivot = weight_log[weight_log["feature"].isin(top_features_p4)].pivot(index="period", columns="feature", values="w_cost")
pivot = pivot.reindex(["P0", "P1", "P2", "P3", "P4"])

fig, ax = plt.subplots(figsize=(10, 6))
pivot.plot(ax=ax, marker="o")
ax.set_ylabel("w_cost (normalized)")
ax.set_title("Counterfactual EL-sensitivity weight across periods, top-8 P4 features")
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. Stability ablation: static / Method A / B / C vs. Weighted Temporal SHAP at each alpha

`alpha=1.0` should reproduce Method A's numbers exactly (it's the same formula with `w_cost`'s contribution zeroed out) — a useful correctness check before trusting the other alphas.

In [ ]:
display_order = ["static", "method_a", "method_b", "method_c",
                  "weighted_temporal_alpha_1.0", "weighted_temporal_alpha_0.75",
                  "weighted_temporal_alpha_0.5", "weighted_temporal_alpha_0.25", "weighted_temporal_alpha_0.0"]
stability_ordered = stability_df.reindex(display_order)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ["cosine_mean", "kendall_tau_mean", "jaccard_at_10_mean"]
colors = ["tab:gray"] * 4 + ["tab:blue"] * 5
for ax, metric in zip(axes, metrics):
    ax.barh(stability_ordered.index[::-1], stability_ordered[metric][::-1], color=colors[::-1])
    ax.set_title(metric)
    ax.axvline(stability_ordered.loc["method_a", metric], color="red", linestyle="--", linewidth=1, label="Method A (== alpha=1.0)")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

print("alpha=1.0 matches Method A exactly:",
      (stability_ordered.loc["weighted_temporal_alpha_1.0"].round(6) == stability_ordered.loc["method_a"].round(6)).all())
stability_ordered.round(4)

## 4. Fairness reduction: SHAP-guided selective threshold correction vs. baseline (== Method B's unchanged decision)

Evaluated only on the 4 planted-bias periods (P1–P4), against the Phase 1 ground-truth `CODE_GENDER` bias. Remember the scale caveat from the script's docstring: this runs on the same 500-applicant eval sample SHAP needs, not Phase 4's full 307K-row population — read these as directional evidence the mechanism works, not a portfolio-scale fairness claim.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fairness_df[["DPD_baseline", "DPD_corrected"]].plot(kind="bar", ax=axes[0], color=["tab:gray", "tab:green"])
axes[0].set_title("Demographic Parity Difference: baseline vs. SHAP-guided correction")
axes[0].set_ylabel("DPD")

fairness_df[["EOD_baseline", "EOD_corrected"]].plot(kind="bar", ax=axes[1], color=["tab:gray", "tab:green"])
axes[1].set_title("Equal Opportunity Difference: baseline vs. SHAP-guided correction")
axes[1].set_ylabel("EOD")

plt.tight_layout()
plt.show()

fairness_df[["n_flagged", "flagged_features", "DeltaDPD", "DeltaEOD"]]

## Notes / what to take away

- If `alpha=1.0`'s stability numbers don't exactly match Method A's, something in the blend/rescale logic regressed — treat that as a correctness bug, not a modeling nuance.
- A positive `DeltaDPD`/`DeltaEOD` means the SHAP-guided correction *reduced* the disparity relative to the unchanged baseline decision (== what Method B would have produced, since it never alters decisions). A near-zero or negative value in any period means the mitigation mechanism didn't help there — report that honestly per period rather than only the average, the same way Phase 5's stability ranking was reported honestly even though it reversed the source paper.
- `flagged_features` shows which features actually triggered the selective correction each period — worth checking these are plausible (e.g. features that are both known to be in `DRIFT_FEATURES` or correlated with them) rather than an artifact.
- This is the ablation and fairness evidence the roadmap's Phase 6 exit criteria ask for: a working implementation, a completed ablation table, and a one-sentence stateable finding (e.g. "cost-aware weighting at alpha=X improves fairness reduction by Y over Method A/baseline at comparable stability") — read off the tables above once run.